In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_snapshot_to_process AS
SELECT min(source_file_date) AS snapshot_date
  FROM tibia_analytics.silver.characters_history
 WHERE source_file_date > (
         SELECT coalesce(max(last_seen_date), DATE '1997-01-07')
           FROM tibia_analytics.silver.characters_identity
);

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_previous_snapshot AS
SELECT max(valid_to) AS previous_snapshot_date
  FROM tibia_analytics.silver.characters_state
 WHERE valid_to < (
         SELECT snapshot_date
           FROM tibia_analytics.silver.tmp_snapshot_to_process
);

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_character AS
WITH raw_characters AS (
   SELECT name,
          world,
          filter(former_names, x -> x != name) AS former_names, -- FILTER TO FIX API ISSUE
          source_file_date AS snapshot_date,
          deletion_at AS deleted_at,
          api_timestamp
     FROM tibia_analytics.silver.characters_history
    WHERE source_file_date = (
           SELECT snapshot_date 
             FROM tibia_analytics.silver.tmp_snapshot_to_process
          )
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY name, 
                         source_file_date
                ORDER BY api_timestamp DESC
  ) = 1
),
names AS (
  SELECT DISTINCT 
         name,
         world,
         former_names,
         snapshot_date,
         deleted_at,
         array_union(coalesce(former_names, array()), array(name)) AS locked_names,
         api_timestamp
    FROM raw_characters
),
filtered_state AS (
  SELECT cs.*
    FROM tibia_analytics.silver.characters_state AS cs
   INNER JOIN tibia_analytics.silver.characters_identity AS ci
      ON ci.character_id = cs.character_id
     AND ci.is_deleted = FALSE
),
all_previous_name AS (
SELECT character_id,
       collect_set(name) AS all_previous_names
  FROM (  
         SELECT character_id,
                name,
                is_active_lock
           FROM filtered_state
        QUALIFY ROW_NUMBER() OVER (
                  PARTITION BY character_id, name
                      ORDER BY valid_from DESC
                ) = 1
       )
 WHERE is_active_lock = TRUE
 GROUP BY character_id
),
previous_snapshot AS (
  SELECT character_id,
         name AS previous_name
    FROM filtered_state
   WHERE name_type = 'observed_current'
     AND valid_from <= (SELECT previous_snapshot_date FROM tibia_analytics.silver.tmp_previous_snapshot)
 QUALIFY ROW_NUMBER() OVER (
           PARTITION BY character_id
               ORDER BY valid_from DESC
         ) = 1
),
direct_match AS (
  SELECT ps.character_id AS direct_match_id,
         n.name,
         n.world,
         n.former_names,
         n.locked_names,
         n.snapshot_date,
         n.deleted_at,
         n.api_timestamp
    FROM names AS n
    LEFT JOIN previous_snapshot AS ps
           ON ps.previous_name = n.name
),
former_match AS (
  SELECT dm.direct_match_id,
         ps.character_id AS former_match_id,
         dm.name,
         dm.world,
         dm.former_names,
         dm.locked_names,
         dm.snapshot_date,
         dm.deleted_at,
         dm.api_timestamp
    FROM direct_match AS dm
    LEFT JOIN previous_snapshot AS ps
           ON dm.direct_match_id IS NULL
          AND array_contains(dm.former_names, ps.previous_name)
),
claimed_ids AS (
  SELECT direct_match_id AS character_id
    FROM direct_match
   WHERE direct_match_id IS NOT NULL
   UNION
  SELECT former_match_id AS character_id
    FROM former_match
   WHERE former_match_id IS NOT NULL
),
historical_match AS (
  SELECT fm.direct_match_id,
         fm.former_match_id,
         CASE
           WHEN claimed.character_id IS NULL THEN apn.character_id 
           ELSE NULL
         END AS historical_match_id,
         fm.name,
         fm.world,
         fm.former_names,
         fm.locked_names,
         fm.snapshot_date,
         fm.deleted_at,
         fm.api_timestamp
    FROM former_match AS fm
    LEFT JOIN all_previous_name AS apn
           ON fm.direct_match_id IS NULL
          AND fm.former_match_id IS NULL
          AND size(array_intersect(fm.locked_names, apn.all_previous_names)) > 0
    LEFT JOIN claimed_ids AS claimed
           ON claimed.character_id = apn.character_id
),
identity_resolution AS (
  SELECT coalesce(
           direct_match_id,
           former_match_id,
           historical_match_id, 
           sha1(
             concat_ws(
               '|',
               name,
               snapshot_date
             )
           )
         ) AS character_id,
         name,
         world,
         former_names,
         locked_names,
         snapshot_date,
         deleted_at,
         api_timestamp
    FROM historical_match
)
SELECT DISTINCT
       character_id,
       name,
       world,
       former_names,
       locked_names,
       snapshot_date,
       deleted_at
  FROM identity_resolution
  QUALIFY ROW_NUMBER() OVER (
            PARTITION BY character_id, 
                         snapshot_date
                ORDER BY api_timestamp DESC
  ) = 1;

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.character_stage AS
SELECT * 
  FROM tmp_character;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_previous_snapshot_character AS
 SELECT character_id,
        MAX(valid_to) AS previous_snapshot_date
   FROM tibia_analytics.silver.characters_state
  WHERE valid_to < (
          SELECT snapshot_date 
            FROM tibia_analytics.silver.tmp_snapshot_to_process
        )
  GROUP BY character_id;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_state AS
WITH observed_current AS (
  SELECT character_id,
         name,
         world,
         'observed_current' AS name_type,
         TRUE AS is_active_lock,
         snapshot_date
    FROM tibia_analytics.silver.character_stage
),
to_classify AS (
  SELECT character_id,
         fn AS name,
         NULL AS world,
         'to_classify' AS name_type,
         TRUE AS is_active_lock,
         snapshot_date
    FROM tibia_analytics.silver.character_stage
 LATERAL VIEW explode(former_names) exploded AS fn
),
current_snapshot AS (
  SELECT *
    FROM observed_current
   UNION ALL
  SELECT *
    FROM to_classify
),
previous_snapshot AS (
  SELECT character_id,
         name AS previous_name
    FROM tibia_analytics.silver.characters_state
   WHERE name_type = 'observed_current'
 QUALIFY ROW_NUMBER() OVER (
           PARTITION BY character_id, 
                        name
               ORDER BY valid_from DESC
         ) = 1
)
SELECT DISTINCT
       cs.character_id,
       cs.name,
       CASE
         WHEN cs.name_type = 'observed_current' THEN cs.world
         ELSE NULL
       END AS world,
       CASE
         WHEN cs.name_type = 'observed_current' THEN cs.name_type
         WHEN ps.character_id IS NOT NULL        THEN 'observed_historical'
         ELSE 'inferred_historical'
       END AS name_type,
       cs.is_active_lock,
       cs.snapshot_date,
       tpsc.previous_snapshot_date
  FROM current_snapshot AS cs
  LEFT JOIN tmp_previous_snapshot_character AS tpsc
    ON tpsc.character_id = cs.character_id
  LEFT JOIN previous_snapshot AS ps
    ON ps.character_id = cs.character_id
   AND ps.previous_name = cs.name;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_identity AS
SELECT *
  FROM tibia_analytics.silver.character_stage;

In [0]:
MERGE INTO tibia_analytics.silver.characters_identity AS target
USING tmp_identity AS source
ON target.character_id = source.character_id
WHEN MATCHED THEN UPDATE SET
  target.current_name   = source.name,
  target.current_world  = source.world,
  target.locked_names   = source.locked_names,
  target.all_names      = array_union(coalesce(target.all_names, array()), source.locked_names),
  target.last_seen_date = source.snapshot_date,
  target.deleted_at     = source.deleted_at,
  target.is_deleted     = CASE
                            WHEN source.deleted_at IS NOT NULL AND source.snapshot_date >= CAST(source.deleted_at AS DATE) THEN TRUE
                            ELSE FALSE
                          END,
  target.updated_at     = current_timestamp(),
  target.processed_at   = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  character_id,
  current_name,
  current_world,
  locked_names,
  all_names,
  first_seen_date,
  last_seen_date,
  deleted_at,
  is_deleted,
  created_at,
  updated_at,
  processed_at
)
VALUES (
  source.character_id,
  source.name,
  source.world,
  source.locked_names,
  source.locked_names,
  source.snapshot_date,
  source.snapshot_date,
  source.deleted_at,
  CASE
    WHEN source.deleted_at IS NOT NULL
     AND source.snapshot_date >= CAST(source.deleted_at AS DATE)
      THEN TRUE
    ELSE FALSE
  END,
  current_timestamp(),
  current_timestamp(),
  current_timestamp()
);

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target
USING tmp_state AS source
ON target.character_id    = source.character_id
AND target.name           = source.name
AND (target.world         = source.world 
    OR target.name_type  != 'observed_current')
AND target.name_type      = source.name_type
AND target.is_active_lock = TRUE
AND target.valid_to       = source.previous_snapshot_date
WHEN MATCHED THEN UPDATE SET
  target.valid_to = source.snapshot_date,
  target.updated_at = current_timestamp(),
  target.processed_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  character_id,
  name,
  world,
  name_type,
  valid_from,
  valid_to,
  is_active_lock,
  created_at,
  updated_at,
  processed_at
)
VALUES (
  source.character_id,
  source.name,
  source.world,
  source.name_type,
  source.snapshot_date,
  source.snapshot_date,
  source.is_active_lock,
  current_timestamp(),
  current_timestamp(),
  current_timestamp()
);

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_last_character_snapshot AS
SELECT character_id,
       MAX(valid_to) AS last_character_snapshot
  FROM tibia_analytics.silver.characters_state
 WHERE is_active_lock = TRUE
 GROUP BY character_id;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_last_name_snapshot AS
 SELECT character_id,
        name,
        world AS last_world,
        name_type,
        valid_from,
        valid_to,
        is_active_lock
   FROM tibia_analytics.silver.characters_state
QUALIFY ROW_NUMBER() OVER(
          PARTITION BY character_id,
                       name
              ORDER BY valid_from DESC
        ) = 1;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_missing_names AS
SELECT ns.character_id,
       ns.name,
       ns.last_world,
       ns.name_type,
       ns.valid_from,
       ns.valid_to AS last_name_snapshot,
       ns.is_active_lock,
       cs.last_character_snapshot,
       DATE_DIFF(cs.last_character_snapshot, ns.valid_to) AS snapshot_gap,
       EXISTS(
          SELECT 1 
            FROM tibia_analytics.silver.characters_state h
           WHERE h.character_id   = ns.character_id
             AND h.name           = ns.name
             AND h.is_active_lock = FALSE
             AND h.valid_from     = DATE_ADD(ns.valid_to, 1)
       ) AS has_historical_record
  FROM tmp_last_name_snapshot AS ns
 INNER JOIN tmp_last_character_snapshot AS cs
    ON cs.character_id   = ns.character_id
 WHERE ns.is_active_lock = TRUE
   AND DATE_DIFF(cs.last_character_snapshot, ns.valid_to) > 7;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_confirmed_missing_names AS
SELECT character_id,
       name,
       last_world,
       name_type,
       DATE_ADD(last_name_snapshot, 1) AS new_valid_from,
       NULL AS new_valid_to,
       FALSE AS new_is_active_lock
  FROM tmp_missing_names
 WHERE NOT has_historical_record

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target 
USING tmp_confirmed_missing_names AS source
   ON target.character_id   = source.character_id
  AND target.name           = source.name
  AND target.valid_from     = source.new_valid_from
  AND target.is_active_lock = FALSE
 WHEN NOT MATCHED THEN
INSERT (
    character_id,
    name,
    world,
    name_type,
    valid_from,
    valid_to,
    is_active_lock,
    created_at,
    updated_at,
    processed_at
)
VALUES (
    source.character_id,
    source.name,
    source.last_world,
    source.name_type,
    source.new_valid_from,
    source.new_valid_to,
    source.new_is_active_lock,
    current_timestamp(),
    current_timestamp(),
    current_timestamp()
);

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_identity_review AS
WITH canonical_id AS (
  SELECT to_keep.character_id AS canonical_id,
         duplicated.character_id AS duplicated_id,
         to_keep.current_name,
         to_keep.current_world,
         to_keep.locked_names,
         LEAST(to_keep.first_seen_date, duplicated.first_seen_date) AS first_seen_date,
         GREATEST(to_keep.last_seen_date, duplicated.last_seen_date) AS last_seen_date
    FROM tibia_analytics.silver.characters_identity AS to_keep
   INNER JOIN tibia_analytics.silver.characters_identity AS duplicated
      ON to_keep.current_name  = duplicated.current_name
     AND to_keep.current_world = duplicated.current_world
     --AND to_keep.locked_names  = duplicated.locked_names -- REMOVED TO FIX ISSUE | UNMEASURED IMPACT
     AND to_keep.character_id <> duplicated.character_id
     AND ABS(DATE_DIFF(to_keep.last_seen_date, duplicated.last_seen_date)) <= 180
     --AND to_keep.is_deleted = FALSE
     --AND duplicated.is_deleted = FALSE
 QUALIFY ROW_NUMBER() OVER(
           PARTITION BY to_keep.current_name, 
                        to_keep.current_world--, 
                        --to_keep.locked_names
               ORDER BY to_keep.created_at
         ) = 1
)
SELECT *
  FROM canonical_id;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_identity_map AS
SELECT canonical_id,
       duplicated_id
  FROM tibia_analytics.silver.tmp_identity_review;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_unified_state AS
SELECT COALESCE(im.canonical_id, cs.character_id) AS character_id,
       cs.name,
       cs.world,
       cs.name_type,
       cs.valid_from,
       cs.valid_to,
       cs.is_active_lock,
       cs.created_at
  FROM tibia_analytics.silver.characters_state AS cs
  LEFT JOIN tmp_identity_map AS im
    ON im.duplicated_id = cs.character_id
 WHERE cs.character_id IN (
         SELECT canonical_id 
           FROM tmp_identity_map
          UNION
         SELECT duplicated_id 
           FROM tmp_identity_map
        );

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_deduplicated_state AS
 SELECT *
   FROM tmp_unified_state
QUALIFY ROW_NUMBER() OVER(
          PARTITION BY character_id,
                       name,
                       COALESCE(world, ''),
                       name_type,
                       valid_from,
                       COALESCE(valid_to, DATE '9999-12-31')
              ORDER BY created_at
        ) = 1;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_state_final AS
SELECT ds.*
  FROM tmp_deduplicated_state AS ds
 WHERE NOT (
       ds.name_type = 'inferred_historical'
       AND EXISTS (
             SELECT 1
               FROM tmp_deduplicated_state AS ds2
              WHERE ds2.character_id = ds.character_id
                AND ds2.name         = ds.name
                AND ds2.name_type    = 'observed_historical'
                AND ds2.valid_from   = ds.valid_from
                AND COALESCE(ds2.valid_to, DATE '9999-12-31')
                    = COALESCE(ds.valid_to, DATE '9999-12-31')
         )
       );

In [0]:
MERGE INTO tibia_analytics.silver.characters_identity AS target
USING tibia_analytics.silver.tmp_identity_review AS source
ON target.character_id   = source.canonical_id
WHEN MATCHED THEN UPDATE SET
  target.first_seen_date = LEAST(target.first_seen_date, source.first_seen_date),
  target.last_seen_date  = GREATEST(target.last_seen_date, source.last_seen_date),
  target.updated_at      = current_timestamp(),
  target.processed_at    = current_timestamp();

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target
USING tmp_identity_map AS source
ON target.character_id = source.duplicated_id
WHEN MATCHED THEN UPDATE SET
  target.character_id = source.canonical_id,
  target.updated_at   = current_timestamp(),
  target.processed_at = current_timestamp();

In [0]:
DELETE FROM tibia_analytics.silver.characters_identity
WHERE character_id IN (
        SELECT duplicated_id
          FROM tmp_identity_map
      );

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_state_to_delete AS
SELECT cs.character_id,
       cs.name,
       cs.world,
       cs.name_type,
       cs.valid_from,
       cs.valid_to
  FROM tibia_analytics.silver.characters_state AS cs
  LEFT JOIN tmp_state_final AS sf
    ON sf.character_id = cs.character_id
   AND sf.name         = cs.name
   AND COALESCE(sf.world,'') = COALESCE(cs.world,'')
   AND sf.name_type    = cs.name_type
   AND sf.valid_from   = cs.valid_from
   AND COALESCE(sf.valid_to, DATE '9999-12-31')
       = COALESCE(cs.valid_to, DATE '9999-12-31')
 WHERE sf.character_id IS NULL
   AND cs.character_id IN (
        SELECT canonical_id
        FROM tmp_identity_map
       );

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target
USING tmp_state_to_delete AS source
ON target.character_id        = source.character_id
AND target.name               = source.name
AND COALESCE(target.world,'') = COALESCE(source.world,'')
AND target.name_type          = source.name_type
AND target.valid_from         = source.valid_from
AND COALESCE(target.valid_to, DATE '9999-12-31')
    = COALESCE(source.valid_to, DATE '9999-12-31')
WHEN MATCHED THEN DELETE;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_identities_to_set_is_deleted_as_true AS
WITH max_date AS (
  SELECT max(last_seen_date) AS last_snapshot
    FROM tibia_analytics.silver.characters_identity
)
SELECT ci.*
  FROM tibia_analytics.silver.characters_identity AS ci
 CROSS JOIN max_date AS md
 WHERE DATE_DIFF(md.last_snapshot, ci.last_seen_date) > 7
   AND ci.deleted_at IS NULL
   AND ci.is_deleted = FALSE;

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_states_to_change AS
WITH base As (
  SELECT *
    FROM tibia_analytics.silver.characters_state
 QUALIFY ROW_NUMBER() OVER(
           PARTITION BY character_id, 
                        name
               ORDER BY valid_from DESC
         ) = 1
),
release_evaluation AS (
  SELECT base.*,
         snapshot.snapshot_date,
         DATEDIFF(snapshot.snapshot_date, base.valid_to) AS days_since_last_seen
    FROM base As base
   CROSS JOIN tibia_analytics.silver.tmp_snapshot_to_process AS snapshot 
)
SELECT *
  FROM release_evaluation
 WHERE is_active_lock = TRUE
   AND days_since_last_seen >= 180;

In [0]:
MERGE INTO tibia_analytics.silver.characters_identity AS target
USING tmp_identities_to_set_is_deleted_as_true AS source
   ON target.character_id = source.character_id
 WHEN MATCHED 
  AND target.deleted_at IS NULL
  AND target.is_deleted     = FALSE
  AND target.last_seen_date = source.last_seen_date 
 THEN UPDATE SET
      target.deleted_at     = to_timestamp(source.last_seen_date),
      target.is_deleted     = TRUE,
      target.updated_at     = current_timestamp(),
      target.processed_at   = current_timestamp();

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target
USING tmp_states_to_change AS source
   ON target.character_id   = source.character_id
  AND target.name           = source.name
  AND (target.world         = source.world 
      OR target.name_type  != 'observed_current')
  AND target.name_type      = source.name_type
  AND target.valid_from = DATE_ADD(source.snapshot_date, 1)
  AND target.is_active_lock = FALSE
 WHEN NOT MATCHED 
 THEN INSERT (
  character_id,
  name,
  world,
  name_type,
  valid_from,
  valid_to,
  is_active_lock,
  created_at,
  updated_at,
  processed_at
)
VALUES (
  source.character_id,
  source.name,
  source.world,
  source.name_type,
  DATE_ADD(source.snapshot_date, 1),
  NULL,
  FALSE,
  current_timestamp(),
  current_timestamp(),
  current_timestamp()
)

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_states_to_unlock_by_reappearance AS
WITH base As (
  SELECT *
    FROM tibia_analytics.silver.characters_state
 QUALIFY ROW_NUMBER() OVER(
           PARTITION BY character_id, 
                        name,
                        name_type
               ORDER BY valid_from DESC
         ) = 1
),
observed_current AS (
  SELECT *
    FROM base
   WHERE name_type = 'observed_current'
),
historical_active AS (
  SELECT *
    FROM base
   WHERE name_type != 'observed_current'
     AND is_active_lock = TRUE
),
states_to_insert AS (
  SELECT ha.*
    FROM historical_active AS ha
   INNER JOIN observed_current AS oc
      ON oc.character_id = ha.character_id
     AND oc.name = ha.name
     AND oc.valid_from > ha.valid_from
)
SELECT *
  FROM states_to_insert;

In [0]:
MERGE INTO tibia_analytics.silver.characters_state AS target
USING tmp_states_to_unlock_by_reappearance AS source
   ON target.character_id   = source.character_id
  AND target.name           = source.name
  AND target.name_type      = source.name_type
  AND target.valid_from     = DATE_ADD(source.valid_to, 1)
  AND target.is_active_lock = FALSE
WHEN NOT MATCHED THEN
INSERT (
  character_id,
  name,
  world,
  name_type,
  valid_from,
  valid_to,
  is_active_lock,
  created_at,
  updated_at,
  processed_at
)
VALUES (
  source.character_id,
  source.name,
  source.world,
  source.name_type,
  DATE_ADD(source.valid_to, 1),
  NULL,
  FALSE,
  current_timestamp(),
  current_timestamp(),
  current_timestamp()
);

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_identities_to_reactivate AS
WITH last_observed_current AS (
  SELECT character_id,
         MAX(valid_from) AS last_seen_date
    FROM tibia_analytics.silver.characters_state
   WHERE name_type = 'observed_current'
   GROUP BY character_id
),
deleted_identities AS (
  SELECT *
    FROM tibia_analytics.silver.characters_identity
   WHERE is_deleted = TRUE
)
SELECT di.character_id,
       loc.last_seen_date
  FROM deleted_identities AS di
 INNER JOIN last_observed_current AS loc
    ON loc.character_id = di.character_id
   AND loc.last_seen_date > di.deleted_at;

In [0]:
MERGE INTO tibia_analytics.silver.characters_identity AS target
USING tmp_identities_to_reactivate AS source
   ON target.character_id = source.character_id
WHEN MATCHED THEN UPDATE SET
  target.is_deleted     = FALSE,
  target.deleted_at     = NULL,
  target.last_seen_date = source.last_seen_date,
  target.updated_at     = current_timestamp(),
  target.processed_at   = current_timestamp();

In [0]:
DROP VIEW IF EXISTS tmp_character;
DROP VIEW IF EXISTS tmp_identity;
DROP VIEW IF EXISTS tmp_previous_snapshot_character;
DROP VIEW IF EXISTS tmp_state;

DROP VIEW IF EXISTS tmp_last_character_snapshot;
DROP VIEW IF EXISTS tmp_last_name_snapshot;
DROP VIEW IF EXISTS tmp_missing_names;
DROP VIEW IF EXISTS tmp_confirmed_missing_names;

DROP VIEW IF EXISTS tmp_identity_map;
DROP VIEW IF EXISTS tmp_unified_state;
DROP VIEW IF EXISTS tmp_deduplicated_state; 
DROP VIEW IF EXISTS tmp_state_final;
DROP VIEW IF EXISTS tmp_state_to_delete;

DROP VIEW IF EXISTS tmp_identities_to_set_is_deleted_as_true;
DROP VIEW IF EXISTS tmp_states_to_change;
DROP VIEW IF EXISTS tmp_states_to_unlock_by_reappearance;
DROP VIEW IF EXISTS tmp_identities_to_reactivate;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_snapshot_to_process;
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_previous_snapshot;
DROP TABLE IF EXISTS tibia_analytics.silver.character_stage;
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_identity_review;